# Connecting to Interactive Brokers TWS and pulling data

This notebook opens a socket connection to a **running** TWS or IB Gateway, then pulls
account summary, contract details, historical bars, snapshot quotes and positions.
The last section writes the bars into this repo's `.screen_cache/` in the exact layout
`data.load_history()` expects, so anything downstream (screeners, backtests) can read
IB data instead of yfinance.

The connection is opened **read only** — TWS will reject order placement on this session
even if a later cell tries. That is deliberate: this is a data notebook.

## Before you run anything

TWS/Gateway must be running and logged in on the *same machine as this notebook* (or
reachable over the network), with the API switched on:

1. TWS → **File ▸ Global Configuration ▸ API ▸ Settings**
   (Gateway → **Configure ▸ Settings ▸ API ▸ Settings**)
2. Tick **Enable ActiveX and Socket Clients**
3. Tick **Read-Only API** (recommended for this notebook)
4. Untick **Allow connections from localhost only** *only* if the notebook runs on
   another machine, and then add that machine's IP under **Trusted IPs**
5. Confirm the **Socket port** matches the table below
6. Apply, and accept the "incoming connection" popup the first time you connect

| Application | Paper | Live |
|---|---|---|
| TWS | `7497` | `7496` |
| IB Gateway | `4002` | `4001` |

Each concurrent API session needs its own `clientId`. Reusing an id that another
script already holds is the most common cause of a silent connect timeout.

**Market data:** historical bars and delayed quotes work on a paper account with no
subscription. Live streaming quotes need the relevant market-data subscription on the
account; without one you get delayed data (15 min) or nothing at all.

## 0. Install and import

In [1]:
# ib_async is the maintained successor to ib_insync. Install once, then restart the kernel:
# %pip install ib_async pandas plotly

try:                     # preferred
    from ib_async import IB, Stock, Forex, Index, Contract, util
    _IB_LIB = "ib_async"
except ImportError:      # older environments still on the original package
    from ib_insync import IB, Stock, Forex, Index, Contract, util
    _IB_LIB = "ib_insync"

import os
from pathlib import Path

import pandas as pd

# Jupyter already owns an asyncio event loop; this lets the synchronous ib.* calls
# below run inside it. Without it every blocking call raises "This event loop is
# already running".
util.startLoop()

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print(f"using {_IB_LIB}")

using ib_async


## 1. Connection settings

Defaults point at **TWS paper** on localhost. Override with environment variables
(`IB_HOST`, `IB_PORT`, `IB_CLIENT_ID`) or just edit the values here.

In [ ]:
IB_HOST = os.getenv("IB_HOST", "127.0.0.1")
IB_PORT = int(os.getenv("IB_PORT", "7497"))   # 7497 TWS paper | 7496 TWS live | 4002 GW paper | 4001 GW live
IB_CLIENT_ID = int(os.getenv("IB_CLIENT_ID", "17"))  # any unused id; 0 is reserved for the manual-order feed
IB_READONLY = True    # keep True: this notebook only reads

print(f"target: {IB_HOST}:{IB_PORT}  clientId={IB_CLIENT_ID}  readonly={IB_READONLY}")

In [ ]:
ib = IB()

try:
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID, readonly=IB_READONLY, timeout=10)
except Exception as exc:
    print(f"connect failed: {type(exc).__name__}: {exc}\n")
    print("Work through these, in order — one of them is almost always it:")
    print("  1. Is TWS/Gateway actually running and *logged in*? A login screen accepts no API calls.")
    print("  2. Does the socket port in Global Configuration > API > Settings match IB_PORT above?")
    print("     Paper and live are different ports; so are TWS and Gateway.")
    print("  3. Is 'Enable ActiveX and Socket Clients' ticked?")
    print("  4. Is another script holding this clientId? Change IB_CLIENT_ID and retry.")
    print("  5. Remote machine: untick 'Allow connections from localhost only' and add the IP")
    print("     under Trusted IPs. A firewall between the two will also time out silently.")
    print("  6. TWS pops up an 'incoming connection' dialog on first connect — accept it.")
else:
    print(f"connected: {ib.isConnected()}")
    print(f"server version : {ib.client.serverVersion()}")
    print(f"server time    : {ib.reqCurrentTime()}")
    print(f"accounts       : {ib.managedAccounts()}")

## 2. Account summary

`accountSummary()` is a one-shot pull of the account's headline values. On a multi-account
login pass the account code explicitly, e.g. `ib.accountSummary('DU1234567')`.

In [ ]:
summary = util.df(ib.accountSummary())

if summary is None or summary.empty:
    print("no account values returned — check the account is funded and logged in")
else:
    summary = summary[["account", "tag", "value", "currency"]]
    headline = [
        "NetLiquidation", "TotalCashValue", "BuyingPower", "GrossPositionValue",
        "AvailableFunds", "ExcessLiquidity", "MaintMarginReq",
    ]
    display(summary[summary["tag"].isin(headline)].reset_index(drop=True))

## 3. Qualify a contract

An IB request is only as good as its contract. `qualifyContracts()` round-trips the stub
you build against IB's database and fills in the `conId`, primary exchange and trading
class. Do this before every data request: an ambiguous ticker (the same symbol listed on
several exchanges) otherwise errors, or worse, silently resolves to the wrong listing.

In [ ]:
SYMBOLS = ["AAPL", "MSFT", "NVDA"]

contracts = [Stock(sym, "SMART", "USD") for sym in SYMBOLS]
ib.qualifyContracts(*contracts)

display(pd.DataFrame([
    {
        "symbol": c.symbol,
        "conId": c.conId,
        "secType": c.secType,
        "exchange": c.exchange,
        "primaryExchange": c.primaryExchange,
        "currency": c.currency,
    }
    for c in contracts
]))

# Non-equity examples, for reference — same pattern:
#   Forex("EURUSD")                                           -> cash
#   Index("SPX", "CBOE")                                      -> index
#   Contract(secType="CONTFUT", symbol="ES", exchange="CME")   -> continuous future

## 4. Historical bars

`reqHistoricalData` is the workhorse. The arguments that matter:

- `durationStr` — how far back: `'60 D'`, `'1 Y'`, `'6 M'`
- `barSizeSetting` — `'1 min'`, `'5 mins'`, `'1 hour'`, `'1 day'`, …
- `whatToShow` — `'TRADES'` for stocks and futures, `'MIDPOINT'` for forex and indices
  (forex has no trade prints, so `'TRADES'` returns an empty list)
- `useRTH` — `True` drops pre- and post-market bars

**Pacing:** IB throttles this endpoint to roughly 60 requests per 10 minutes, and rejects
identical requests fired within 15 seconds. Loop over symbols with a small sleep, cache
what you get, and do not re-pull inside a tight loop.

In [ ]:
OHLCV = ["Open", "High", "Low", "Close", "Volume"]


def fetch_bars(contract, duration="1 Y", bar_size="1 day", what="TRADES", rth=True):
    """Historical bars as an OHLCV frame with a DatetimeIndex.

    Column names and dtypes match what ``data.load_history`` produces, so the
    result is a drop-in for anything in this repo that consumes price history.
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime="",          # empty string = up to now
        durationStr=duration,
        barSizeSetting=bar_size,
        whatToShow=what,
        useRTH=rth,
        formatDate=1,
    )
    if not bars:
        return pd.DataFrame(columns=OHLCV)

    frame = util.df(bars).rename(columns={
        "open": "Open", "high": "High", "low": "Low",
        "close": "Close", "volume": "Volume",
    })
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.set_index("date").sort_index()
    frame.index.name = None
    if getattr(frame.index, "tz", None) is not None:   # intraday bars come back tz-aware
        frame.index = frame.index.tz_localize(None)
    return frame[OHLCV].apply(pd.to_numeric, errors="coerce").dropna(how="any")


daily = fetch_bars(contracts[0], duration="1 Y", bar_size="1 day")
print(f"{contracts[0].symbol}: {len(daily)} daily bars, "
      f"{daily.index.min():%Y-%m-%d} -> {daily.index.max():%Y-%m-%d}")
display(daily.tail())

In [ ]:
# Intraday works the same way — just a shorter duration and a smaller bar size.
intraday = fetch_bars(contracts[0], duration="2 D", bar_size="5 mins")
print(f"{len(intraday)} five-minute bars")
display(intraday.tail())

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(go.Candlestick(
    x=daily.index, open=daily["Open"], high=daily["High"],
    low=daily["Low"], close=daily["Close"], name=contracts[0].symbol,
))
fig.update_layout(
    title=f"{contracts[0].symbol} — 1 year of daily bars from IB",
    xaxis_rangeslider_visible=False, height=460, margin=dict(l=40, r=20, t=50, b=30),
)
fig.show()

## 5. Snapshot quotes

`reqMktData` opens a streaming subscription; the `Ticker` object it hands back mutates in
place as ticks arrive, so you wait a moment and then read it.

`reqMarketDataType(3)` asks for **delayed** data, which is what an account without a
market-data subscription gets. Set it to `1` if you have live subscriptions. Fields you
are not entitled to stay `nan` — that is an entitlement problem, not a bug in the code.

In [ ]:
# 1 = live | 2 = frozen (last close) | 3 = delayed | 4 = delayed-frozen
ib.reqMarketDataType(int(os.getenv("IB_MARKET_DATA_TYPE", "3")))

tickers = [ib.reqMktData(c, "", snapshot=False) for c in contracts]
ib.sleep(3)   # let the first ticks land; ib.sleep pumps the event loop, time.sleep does not

quotes = pd.DataFrame([
    {
        "symbol": t.contract.symbol,
        "bid": t.bid, "bidSize": t.bidSize,
        "ask": t.ask, "askSize": t.askSize,
        "last": t.last, "prevClose": t.close,
        "marketPrice": t.marketPrice(),   # best available: last, else midpoint, else close
        "time": t.time,
    }
    for t in tickers
])
display(quotes)

for c in contracts:      # always cancel: open streams count against the ~100 line limit
    ib.cancelMktData(c)

## 6. Positions and portfolio

`positions()` is the raw book. `portfolio()` adds IB's own valuation — market price,
market value and unrealised PnL — for the account this session is subscribed to. Both
come back empty on a fresh paper account that has never traded.

In [ ]:
positions = util.df(ib.positions())
if positions is None or positions.empty:
    print("no open positions")
else:
    positions["symbol"] = positions["contract"].apply(lambda c: c.symbol)
    display(positions[["account", "symbol", "position", "avgCost"]])

portfolio = util.df(ib.portfolio())
if portfolio is None or portfolio.empty:
    print("no portfolio items")
else:
    portfolio["symbol"] = portfolio["contract"].apply(lambda c: c.symbol)
    display(portfolio[["symbol", "position", "marketPrice", "marketValue",
                       "averageCost", "unrealizedPNL", "realizedPNL"]])

## 7. Feed the bars into this repo's cache

`data.load_history(symbol, period)` reads `./.screen_cache/bt_<SYMBOL>_<period>.csv`
before it ever reaches for yfinance. Writing IB bars to that path swaps the data source
for the screeners and backtests without touching a line of their code.

This is worth doing precisely because yfinance is the weak link — it restates history
after splits and dividends, and returns empty frames when throttled. IB's bars are the
same ones the account trades against.

In [ ]:
import sys

# Locate the repo root (this notebook lives in notebooks/) so `import data` works.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data.py").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import data as repo_data

PERIOD = "1y"   # the period label load_history() will later be called with


def cache_for_repo(frame, symbol, period=PERIOD):
    """Write an OHLCV frame where ``data.load_history`` looks for it."""
    if frame.empty:
        raise ValueError(f"refusing to cache an empty frame for {symbol}")
    repo_data.CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = repo_data.CACHE_DIR / repo_data._cache_path(symbol, period).name
    frame.to_csv(path)
    return path


path = cache_for_repo(daily, contracts[0].symbol)
print(f"wrote {len(daily)} bars -> {path}")

# Round-trip it back through the repo's own loader to prove the format is right.
reloaded = repo_data.load_history(contracts[0].symbol, period=PERIOD)
print(f"load_history returned {len(reloaded)} rows, columns={list(reloaded.columns)}")
display(reloaded.tail(3))

In [ ]:
# Pull the rest of the symbols and cache them too. The sleep is the pacing limit talking:
# ~60 historical requests per 10 minutes, so keep batches small.
for contract in contracts[1:]:
    frame = fetch_bars(contract, duration="1 Y", bar_size="1 day")
    if frame.empty:
        print(f"{contract.symbol}: no bars returned")
        continue
    print(f"{contract.symbol}: {len(frame)} bars -> {cache_for_repo(frame, contract.symbol)}")
    ib.sleep(2)

## 8. Disconnect

Always disconnect. An abandoned session keeps its `clientId` occupied until TWS notices,
and the next run of this notebook will then hang on connect.

In [ ]:
ib.disconnect()
print(f"connected: {ib.isConnected()}")

---

## Notes and gotchas

**`ib.sleep()`, not `time.sleep()`.** IB's client is asynchronous underneath. `time.sleep`
blocks the event loop, so no ticks arrive while you wait — you sleep, then read the same
empty ticker. `ib.sleep` pumps the loop.

**Errors arrive as events, not exceptions.** A request can "succeed" and return nothing
while TWS logs the reason. Subscribe to see them:

```python
ib.errorEvent += lambda reqId, code, msg, contract: print(f"[{code}] {msg}")
```

Codes worth recognising: `162` historical data service error (often pacing),
`200` no security definition (bad contract), `354` not subscribed to market data,
`10197` no market data during a competing live session. `2104`, `2106` and `2158` are
connection-OK notices, not failures.

**Pacing.** ~50 messages/second overall, ~60 historical-data requests per 10 minutes, and
no identical historical request within 15 seconds. Cache aggressively — that is what
section 7 is for.

**One live session per login.** Logging into the same IB account in TWS elsewhere kicks
this session's market data (`10197`). Paper and live are separate logins.

**Keep it read only.** `readonly=True` in section 1 is what stops a stray cell from
sending an order. If you later remove it in order to trade, point at the paper port first
and leave it there until the logic is proven.